
# Reto de clase: Sistema multiagente de soporte con CrewAI + Hugging Face

Este notebook implementa un reto corto de clase usando un dataset real de Hugging Face para construir un sistema multiagente de soporte al cliente.

## Objetivo del reto
Construir un crew que sea capaz de:
1. Analizar un ticket de soporte.
2. Recuperar contexto desde una base local construida a partir del dataset.
3. Redactar una respuesta útil y profesional.
4. Revisar la calidad de la respuesta final.

## Dataset usado
En este notebook se usa el dataset:
**`Tobi-Bueck/customer-support-tickets`**

y se toma un subconjunto pequeño para que el ejercicio sea viable en una sesión de clase.

## Estructura del crew
- Agente 1: Analista de ticket
- Agente 2: Investigador documental (RAG local)
- Agente 3: Redactor de soporte
- Agente 4: Revisor de calidad


## 1. Instalación

In [ ]:
!pip -q install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29 datasets pandas

## 2. Configuración

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = 'sk-xxx'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = 'xxx'

print("Modelo configurado:", os.environ["OPENAI_MODEL_NAME"])


## 3. Carga del dataset desde Hugging Face

In [ ]:

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
print(dataset)


## 4. Explorar un subconjunto pequeño

In [ ]:
split_name = list(dataset.keys())[0]
df = dataset[split_name].to_pandas()

print("Columnas disponibles:", list(df.columns))
df.head(5)


## 5. Construcción de una mini base documental local (RAG sencillo)

In [ ]:
import os

os.makedirs("base_tickets", exist_ok=True)
subset = df.head(20).copy()

def safe_get(row, possible_cols, default=""):
    for c in possible_cols:
        if c in row and pd.notna(row[c]):
            return str(row[c])
    return default

for i, row in subset.iterrows():
    issue = safe_get(row, ["ticket", "instruction", "question", "query", "text", "customer_request"], default="Sin contenido")
    response = safe_get(row, ["response", "answer", "output", "resolution"], default="Sin respuesta")
    category = safe_get(row, ["category", "type", "topic"], default="General")
    priority = safe_get(row, ["priority", "urgency"], default="No especificada")

    content = f"""Categoría: {category}
Prioridad: {priority}

Ticket:
{issue}

Respuesta de referencia:
{response}
"""

    with open(f"base_tickets/ticket_{i}.txt", "w", encoding="utf-8") as f:
        f.write(content)

print("Archivos creados:", len(os.listdir("base_tickets")))
print(os.listdir("base_tickets")[:5])


## 6. Importaciones de CrewAI

In [ ]:

from crewai import Agent, Task, Crew
from crewai_tools import DirectoryReadTool, FileReadTool
from IPython.display import Markdown, display


## 7. Definición de tools

In [ ]:

herramienta_directorio = DirectoryReadTool(directory="base_tickets")
herramienta_archivo = FileReadTool()


## 8. Definición de agentes

In [ ]:
analista_ticket = Agent(
    role="Analista de Ticket de Soporte",
    goal="Comprender el problema del cliente e identificar su intención, severidad y necesidad principal",
    backstory=(
        "Eres un analista de soporte con experiencia en clasificación de incidencias. "
        "Tu trabajo es leer el ticket del cliente, identificar el problema central, "
        "resumirlo y proponer qué tipo de información debería buscarse para responder mejor."
    ),
    allow_delegation=False,
    verbose=True
)

investigador_rag = Agent(
    role="Investigador Documental de Soporte",
    goal="Recuperar casos y respuestas relevantes desde la base documental local",
    backstory=(
        "Eres un agente especializado en recuperación de conocimiento. "
        "Tu función es revisar la base local de tickets, encontrar ejemplos parecidos "
        "y extraer patrones útiles para ayudar a responder el ticket actual."
    ),
    tools=[herramienta_directorio, herramienta_archivo],
    allow_delegation=False,
    verbose=True
)

redactor_soporte = Agent(
    role="Redactor de Respuesta de Soporte",
    goal="Redactar una respuesta clara, útil y profesional para el cliente",
    backstory=(
        "Eres un especialista en comunicación de soporte al cliente. "
        "Debes redactar una respuesta bien estructurada, empática y accionable, "
        "basada en el análisis del ticket y la evidencia recuperada."
    ),
    allow_delegation=False,
    verbose=True
)

revisor_calidad = Agent(
    role="Revisor de Calidad de Soporte",
    goal="Asegurar que la respuesta final sea correcta, completa y profesional",
    backstory=(
        "Eres un revisor senior de calidad. "
        "Tu trabajo es validar claridad, exactitud, tono y cobertura completa del problema."
    ),
    allow_delegation=False,
    verbose=True
)


## 9. Definición de tareas

In [ ]:

tarea_analisis = Task(
    description=(
        "Analiza el siguiente ticket de soporte:\n\n"
        "{ticket_cliente}\n\n"
        "Debes identificar:\n"
        "1. Problema principal\n"
        "2. Posible categoría\n"
        "3. Nivel de urgencia\n"
        "4. Qué información sería útil recuperar para responder mejor"
    ),
    expected_output=(
        "Un análisis breve en español del ticket, con diagnóstico inicial, "
        "categoría estimada, nivel de urgencia y necesidades de información."
    ),
    agent=analista_ticket
)

tarea_recuperacion = Task(
    description=(
        "Usa la base documental local para recuperar ejemplos, tickets similares y respuestas de referencia "
        "que puedan ayudar a contestar este caso:\n\n"
        "{ticket_cliente}\n\n"
        "Resume los hallazgos más útiles en español."
    ),
    expected_output=(
        "Un resumen documental en español con ejemplos relevantes y buenas prácticas "
        "extraídas desde la base local."
    ),
    agent=investigador_rag
)

tarea_redaccion = Task(
    description=(
        "Usa el análisis del ticket y la recuperación documental para redactar una respuesta final al cliente. "
        "La respuesta debe ser:\n"
        "- clara\n"
        "- profesional\n"
        "- útil\n"
        "- empática\n"
        "- accionable\n"
        "Debe estar escrita completamente en español."
    ),
    expected_output=(
        "Una respuesta de soporte bien redactada en español, lista para enviar al cliente."
    ),
    agent=redactor_soporte
)

tarea_revision = Task(
    description=(
        "Revisa la respuesta generada por el agente de redacción y optimízala si es necesario. "
        "Asegúrate de que responda completamente al problema del cliente y tenga calidad profesional."
    ),
    expected_output=(
        "Una versión final mejorada de la respuesta de soporte en español."
    ),
    agent=revisor_calidad
)


## 10. Construcción del crew

In [ ]:

equipo_soporte = Crew(
    agents=[analista_ticket, investigador_rag, redactor_soporte, revisor_calidad],
    tasks=[tarea_analisis, tarea_recuperacion, tarea_redaccion, tarea_revision],
    verbose=2,
    memory=True
)


## 11. Selección de un ticket real del dataset

In [ ]:
fila = subset.iloc[0].to_dict()

def extract_ticket_text(row):
    for c in ["ticket", "instruction", "question", "query", "text", "customer_request"]:
        if c in row and pd.notna(row[c]):
            return str(row[c])
    return "No se encontró texto del ticket."

ticket_cliente = extract_ticket_text(fila)
print(ticket_cliente)


## 12. Ejecutar el reto

In [ ]:
resultado = equipo_soporte.kickoff(inputs={"ticket_cliente": ticket_cliente})

## 13. Mostrar resultado

In [ ]:
display(Markdown(resultado))

## 14. Variante: probar con otro ticket del dataset

In [ ]:

fila_2 = subset.iloc[5].to_dict()
ticket_cliente_2 = extract_ticket_text(fila_2)

resultado_2 = equipo_soporte.kickoff(inputs={"ticket_cliente": ticket_cliente_2})
display(Markdown(resultado_2))



## 15. Reto para los estudiantes

### Meta mínima
Lograr que el sistema:
- lea un ticket real del dataset,
- consulte la base local,
- redacte una respuesta,
- y la revise.

### Extensiones opcionales
1. Agregar clasificación automática por categoría.
2. Agregar salida en JSON además de texto.
3. Comparar respuesta:
   - sin RAG
   - con RAG
4. Crear un agente extra:
   - verificador de tono
   - verificador de cumplimiento



## 16. Reflexión final

Este reto integra:
- dataset real de Hugging Face,
- herramientas de CrewAI,
- RAG local simple,
- múltiples agentes especializados,
- workflow secuencial,
- revisión de calidad.

### Mensaje clave
Un crew bien diseñado no solo responde tickets:
**analiza, recupera contexto, sintetiza, redacta y mejora la respuesta final**.
